# Standard QML — 5-Seed Stability Check

Run Standard QML with seeds 42–46 using the same fixed observation-level split and preprocessing as the PAQFE-v2 experiment. Only initialization/training seed changes.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEEDS = [42, 43, 44, 45, 46]
TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
N_QUBITS = 10
N_LAYERS = 3
LR = 0.02
N_EPOCHS = 300
STANDARD_FEATURE_ORDER = ['xAl','VEC','dHmix','dSmix','delta','Omega','dchi','Mean_Tm','Temperature_C','log10_Strain_rate']
TARGET = 'UTS_MPa'
composition_cols = ['Al_norm','Co_norm','Cr_norm','Fe_norm','Ni_norm']


## Load the same Excel workbook and `B_Model_Ready` sheet.

In [ ]:
from google.colab import files
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
df = pd.read_excel(file_path, sheet_name='B_Model_Ready')
cols = STANDARD_FEATURE_ORDER + [TARGET] + composition_cols
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols).copy()
df['Composition_Group'] = df[composition_cols].round(3).astype(str).agg('_'.join, axis=1)
print('Dataset shape:', df.shape)
print('Observations:', len(df))
print('Unique compositions:', df['Composition_Group'].nunique())


In [ ]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SPLIT_RANDOM_STATE)
X_train_raw = train_df[STANDARD_FEATURE_ORDER].to_numpy(dtype=np.float64)
X_test_raw = test_df[STANDARD_FEATURE_ORDER].to_numpy(dtype=np.float64)
y_train_raw = train_df[TARGET].to_numpy(dtype=np.float64)
y_test_raw = test_df[TARGET].to_numpy(dtype=np.float64)
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train_raw)
X_test = x_scaler.transform(X_test_raw)
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1,1)).ravel()
print('Train:', len(X_train), 'Test:', len(X_test))


In [ ]:
def build_standard_qml():
    dev = qml.device('default.qubit', wires=N_QUBITS)
    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def circuit(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
        qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
    class StandardQML(nn.Module):
        def __init__(self):
            super().__init__()
            self.weights = nn.Parameter(0.01 * torch.randn(weight_shape, dtype=torch.float64))
            self.head = nn.Linear(N_QUBITS, 1, dtype=torch.float64)
        def forward(self, x):
            q = torch.stack([circuit(sample, self.weights) for sample in x], dim=0)
            return self.head(q).squeeze(-1)
    return StandardQML()


In [ ]:
Xtr = torch.tensor(X_train, dtype=torch.float64)
ytr = torch.tensor(y_train, dtype=torch.float64)
Xte = torch.tensor(X_test, dtype=torch.float64)
seed_results = []

for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = build_standard_qml()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    model.train()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        pred = model(Xtr)
        loss = loss_fn(pred, ytr)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        pred_std = model(Xte).cpu().numpy()
    pred = y_scaler.inverse_transform(pred_std.reshape(-1,1)).ravel()
    mae = mean_absolute_error(y_test_raw, pred)
    rmse = np.sqrt(mean_squared_error(y_test_raw, pred))
    r2 = r2_score(y_test_raw, pred)
    seed_results.append({'Seed':seed, 'MAE':mae, 'RMSE':rmse, 'R2':r2})
    print(f'Seed {seed}: MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.3f}')


In [ ]:
seed_results_df = pd.DataFrame(seed_results)
summary = pd.DataFrame({
    'Metric':['MAE','RMSE','R2'],
    'Mean':[seed_results_df.MAE.mean(), seed_results_df.RMSE.mean(), seed_results_df.R2.mean()],
    'SD':[seed_results_df.MAE.std(ddof=1), seed_results_df.RMSE.std(ddof=1), seed_results_df.R2.std(ddof=1)]
})
display(seed_results_df)
display(summary)
seed_results_df.to_csv('Standard_QML_seed_stability_results.csv', index=False)
summary.to_csv('Standard_QML_seed_stability_summary.csv', index=False)


## Stop here

Compare this matched 5-seed result with PAQFE-v2: MAE 217.64 ± 16.73, RMSE 273.53 ± 23.89, R² 0.222 ± 0.134. Do not tune either model from this test-set result.